# PARC2026 — M2/M3 Bring-up Smoke (CPU/L4/A100)

`70_model_benchmark_a100.ipynb` の前に、GPU学習を開始せずにD10・Drive・model registry・upstream pin・OpenVLA RLDS bridge readinessを確認する軽量smokeです。

- CPUでも実行可。L4/A100ならGPU情報も記録。
- 大規模モデル重みはロードしない。
- dataset 15GBのローカルstageはしない。
- M3学習は絶対に開始しない。
- `SMOKE_READY_FOR_70` か、具体的なblock理由をDriveへ保存します。


In [ ]:
# 0/4 Drive + D10 + environment
import os, json, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/parc2026-cache')
OUT = DRIVE/'model-benchmark-smoke-v1'
OUT.mkdir(parents=True, exist_ok=True)

try:
    tok = os.environ.get('HF_TOKEN') or userdata.get('HF_TOKEN')
except Exception:
    tok = None
if not tok:
    raise RuntimeError('Colab Secrets に HF_TOKEN を登録してください。')
os.environ['HF_TOKEN'] = tok

try:
    gpu = subprocess.check_output(
        ['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],
        text=True
    ).strip()
except Exception:
    gpu = 'CPU / no nvidia-smi'

free = shutil.disk_usage('/content').free/1024**3
print('device:', gpu)
print(f'local free: {free:.1f} GiB')

decision_path = DRIVE/'pi05-top2-tiebreak-v1/provisional_best_dataset_recipe.json'
assert decision_path.exists(), decision_path
decision = json.loads(decision_path.read_text())
assert decision.get('status') == 'DECIDED', decision
assert decision.get('selected_variant') == 'V2_SQRT_BALANCED_RAW', decision
print('D10:', decision['selected_variant'], 'seed=', decision['eval_seed'])
print('=== D10 GATE: PASS ===')


In [ ]:
# 1/4 Repo + registry + prefetched dataset metadata
import json, subprocess
from pathlib import Path

ROOT = Path('/content/parc2026')
REPO = ROOT/'py_AI'
ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--force','origin/main'],check=True)

registry_path = REPO/'experiments/model_registry_v1.json'
assert registry_path.exists(), (
    f'{registry_path} missing. PR #32 must be merged before running this smoke.'
)
registry = json.loads(registry_path.read_text())
names = [m['id'] for m in registry['models']]
assert names == ['pi05','smolvla','openvla_oft'], names
print('model registry:', names)

src = DRIVE/'datasets/lerobot_libero_plus_v3_train'
for p in [src/'meta/info.json', src/'meta/stats.json', src/'.parc_prefetch_complete.json']:
    assert p.exists(), p
stats = json.loads((src/'meta/stats.json').read_text())
for f in ['observation.state','action']:
    assert 'q01' in stats[f] and 'q99' in stats[f], f
print('Drive dataset metadata/q01/q99: PASS')
print('=== COMMON INPUT PREFLIGHT: PASS ===')


In [ ]:
# 2/4 Upstream pin smoke — clone source only, no model weights
import subprocess
from pathlib import Path

VENDOR = ROOT/'vendor'
VENDOR.mkdir(parents=True, exist_ok=True)

pins = {
    'smolvla': {
        'url':'https://github.com/huggingface/lerobot.git',
        'sha':'3f2c29ef7e44b1ddccbcda3b6a63939e53639e9e',
        'path':VENDOR/'lerobot-smolvla',
        'required':'src/lerobot',
    },
    'openvla_oft': {
        'url':'https://github.com/small-zeng/openvla-oft.git',
        'sha':'e4287e94541f459edc4feabc4e181f537cd569a8',
        'path':VENDOR/'openvla-oft',
        'required':'vla-scripts/finetune.py',
    },
}

for name,cfg in pins.items():
    path=cfg['path']
    if not (path/'.git').exists():
        subprocess.run(['git','init','-q',str(path)],check=True)
        subprocess.run(['git','-C',str(path),'remote','add','origin',cfg['url']],check=True)
    subprocess.run(['git','-C',str(path),'fetch','-q','--depth','1','origin',cfg['sha']],check=True)
    subprocess.run(['git','-C',str(path),'checkout','-q','--force','FETCH_HEAD'],check=True)
    got=subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True).strip()
    assert got == cfg['sha'], (name,got,cfg['sha'])
    assert (path/cfg['required']).exists(), (name,cfg['required'])
    print(name, '@', got, 'PASS')

print('=== UPSTREAM PIN SMOKE: PASS ===')


In [ ]:
# 3/4 Readiness report for notebook 70
import json
from pathlib import Path

rlds = DRIVE/'openvla-rlds-selected-v1'
rlds_contract = rlds/'conversion_contract.json'

blockers = []
if not rlds_contract.exists():
    blockers.append('openvla_selected_subset_rlds_missing')
else:
    c = json.loads(rlds_contract.read_text())
    if c.get('selected_dataset_variant') != 'V2_SQRT_BALANCED_RAW':
        blockers.append('openvla_rlds_variant_mismatch')

# Notebook 70 intentionally still requires these to be frozen explicitly.
blockers += [
    'equal_data_sample_budget_not_frozen',
    'equal_wall_time_budget_not_frozen',
]

report = {
    'schema_version': 1,
    'stage': 'M2_M3_bringup_smoke',
    'd10_status': 'PASS',
    'selected_dataset_variant': 'V2_SQRT_BALANCED_RAW',
    'model_registry': ['pi05','smolvla','openvla_oft'],
    'drive_dataset_metadata': 'PASS',
    'upstream_pins': 'PASS',
    'openvla_selected_rlds_contract': 'PASS' if rlds_contract.exists() else 'MISSING',
    'blockers_before_M3': blockers,
    'status': 'SMOKE_READY_FOR_70' if not blockers else 'EXPECTED_BLOCKERS_REMAIN',
    'note': 'No training or model-weight loading was performed.'
}
(OUT/'smoke_report.json').write_text(json.dumps(report,indent=2)+'\n')
print(json.dumps(report,indent=2))

if blockers:
    print('\n=== SMOKE PASS; 70 SHOULD NOT START M3 YET ===')
    print('Next implementation: exact selected-subset LeRobot -> RLDS bridge, then freeze comparison budgets.')
else:
    print('\n=== SMOKE_READY_FOR_70 ===')
